In [ ]:
import sys
sys.path.append('..')

In [8]:
import os

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.types import Integer, Numeric, String, Boolean
from dotenv import load_dotenv

In [18]:
from pathlib import Path 
from datetime import datetime, timezone

In [9]:
# ============================================================
# 1. DATABASE CONFIGURATION
# ============================================================
load_dotenv()

USER_NAME = os.getenv("USER_NAME")
PASSWORD = os.getenv("PASSWORD")
HOST = os.getenv("HOST")
PORT = os.getenv("PORT")
DB_NAME = os.getenv("DB_NAME")

DATABASE_URL = rf"postgresql+psycopg2://{USER_NAME}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)


In [19]:
# ============================================================ 
# 3. GENERATE RUN INFORMATION 
#  ============================================================ 
# Unique identifier for this ETL execution. 
run_id = datetime.now(timezone.utc).strftime( "%Y%m%d%H%M%S%f" ) 
run_datetime = datetime.now(timezone.utc) 


# script_path = Path(__file__).resolve() 
script_path  = Path.cwd()
script_directory = script_path.parent

Original shape: (7839, 13)
Original columns:
['Week', 'Product', 'Sales_Units', 'Cost_of_Goods_Sold_COGS', 'Inventory_Level', 'Demand_Forecast', 'Price_Per_Unit', 'Gross_Profit', 'Operating_Profit', 'Revenue', 'Inventory_Turnover_Ratio', 'Gross_Profit_Margin', 'Revenue_Growth']


Column validation: PASSED


In [21]:
# Create schemas
with engine.begin() as conn:

    conn.execute(text("""
        CREATE SCHEMA IF NOT EXISTS etl;
    """))
    conn.execute(text("""
        CREATE SCHEMA IF NOT EXISTS raw;
    """))

    conn.execute(text("""
        CREATE SCHEMA IF NOT EXISTS staging;
    """))

    conn.execute(text("""
        CREATE SCHEMA IF NOT EXISTS warehouse;
    """))
    conn.execute(text(""" 
        CREATE TABLE IF NOT EXISTS etl.run_table ( run_id VARCHAR(50) PRIMARY KEY, run_datetime TIMESTAMPTZ NOT NULL, python_script_path TEXT NOT NULL, python_script_directory TEXT NOT NULL ); """)
        )


In [23]:
with engine.begin() as conn:
    conn.execute(text(
        """ INSERT INTO etl.run_table ( run_id, run_datetime, python_script_path, python_script_directory )
        VALUES ( :run_id, :run_datetime, :python_script_path, :python_script_directory )
        """),{
            "run_id": run_id, "run_datetime": run_datetime, "python_script_path": str(script_path), "python_script_directory": str(script_directory)
        })

In [24]:
# ============================================================
# 2. FILE CONFIGURATION
# ============================================================

INP_FILE = "Sample_Data.xlsx"
# ============================================================
# 3. EXPECTED COLUMNS
# ============================================================

EXPECTED_COLUMNS = [
"Week",
"Product",
"Sales_Units",
"Cost_of_Goods_Sold_COGS",
"Inventory_Level",
"Demand_Forecast",
"Price_Per_Unit",
"Gross_Profit",
"Operating_Profit",
"Revenue",
"Inventory_Turnover_Ratio",
"Gross_Profit_Margin",
"Revenue_Growth"
]

# ============================================================
# 4. READ SOURCE DATA
# ============================================================

df = pd.read_excel(INP_FILE)

print("Original shape:", df.shape)
print("Original columns:")
print(df.columns.tolist())


Original shape: (7839, 13)
Original columns:
['Week', 'Product', 'Sales_Units', 'Cost_of_Goods_Sold_COGS', 'Inventory_Level', 'Demand_Forecast', 'Price_Per_Unit', 'Gross_Profit', 'Operating_Profit', 'Revenue', 'Inventory_Turnover_Ratio', 'Gross_Profit_Margin', 'Revenue_Growth']


In [25]:
# ============================================================
# 5. RAW LAYER
# ============================================================

# Check columns before doing any transformation.
if list(df.columns) != EXPECTED_COLUMNS:
    missing_columns = set(EXPECTED_COLUMNS) - set(df.columns)
    extra_columns = set(df.columns) - set(EXPECTED_COLUMNS)

    raise ValueError(
        f"Column validation failed.\n"
        f"Missing columns: {missing_columns}\n"
        f"Extra columns: {extra_columns}"
    )

print("Column validation: PASSED")

Column validation: PASSED


In [26]:
df["run_id"] = run_id
# Save raw data exactly as received.
#
# The raw layer is intentionally not type-converted.

df.to_sql(
    name="sales_data",
    con=engine,
    schema="raw",
    if_exists="replace",
    index=False
)

print("Raw layer loaded successfully.")
print("Raw rows:", len(df))

Raw layer loaded successfully.
Raw rows: 7839


In [28]:

# ============================================================
# 6. STAGING LAYER
# ============================================================

staging_df = df.copy()


# ------------------------------------------------------------
# Standardize column names
# ------------------------------------------------------------

staging_df.columns = [
    "week",
    "product",
    "sales_units",
    "cost_of_goods_sold_cogs",
    "inventory_level",
    "demand_forecast",
    "price_per_unit",
    "gross_profit",
    "operating_profit",
    "revenue",
    "inventory_turnover_ratio",
    "gross_profit_margin",
    "revenue_growth",
    'run_id'
]


In [29]:

# ------------------------------------------------------------
# Convert empty strings to NULL/NaN
# ------------------------------------------------------------

staging_df = staging_df.replace(
    r"^\s*$",
    pd.NA,
    regex=True
)


In [30]:

# ------------------------------------------------------------
# Convert numeric columns
# ------------------------------------------------------------

numeric_columns = [
    "week",
    "sales_units",
    "cost_of_goods_sold_cogs",
    "inventory_level",
    "demand_forecast",
    "price_per_unit",
    "gross_profit",
    "operating_profit",
    "revenue",
    "inventory_turnover_ratio",
    "gross_profit_margin",
    "revenue_growth"
]

for column in numeric_columns:
    staging_df[column] = pd.to_numeric(
        staging_df[column],
        errors="coerce"
    )


# Product should remain a string.
staging_df["product"] = staging_df["product"].astype("string")


In [31]:

# ------------------------------------------------------------
# Remove duplicates based on Week + Product
# ------------------------------------------------------------

before_dedup = len(staging_df)

staging_df = staging_df.drop_duplicates(
    subset=["week", "product"],
    keep="first"
)

after_dedup = len(staging_df)

print(
    f"Removed {before_dedup - after_dedup} duplicate rows."
)


Removed 39 duplicate rows.


In [ ]:

# ------------------------------------------------------------
# Save staging layer
# ------------------------------------------------------------

staging_df.to_sql(
    name="sales_data",
    con=engine,
    schema="staging",
    if_exists="replace",
    index=False,
    dtype={
        "week": Integer(),
        "product": String(100),
        "sales_units": Numeric(),
        "cost_of_goods_sold_cogs": Numeric(),
        "inventory_level": Numeric(),
        "demand_forecast": Numeric(),
        "price_per_unit": Numeric(),
        "gross_profit": Numeric(),
        "operating_profit": Numeric(),
        "revenue": Numeric(),
        "inventory_turnover_ratio": Numeric(),
        "gross_profit_margin": Numeric(),
        "revenue_growth": Numeric(),
        "run_id": String(50)
    }
)

print("Staging layer loaded successfully.")
print("Staging rows:", len(staging_df))


Staging layer loaded successfully.
Staging rows: 7800


In [46]:
with engine.begin() as conn:
    # Product dimension
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS warehouse.dim_product 
        ( product_key SERIAL PRIMARY KEY, product VARCHAR(100) NOT NULL UNIQUE )
                """))
    
    # Week dimension
    conn.execute(text("""
CREATE TABLE IF NOT EXISTS warehouse.dim_week 
( week_key INTEGER PRIMARY KEY, week INTEGER NOT NULL UNIQUE, start_date DATE NOT NULL )
                      """))
    
    # Single warehouse fact table
    conn.execute(text("""
CREATE TABLE IF NOT EXISTS warehouse.sales_data ( sales_key SERIAL PRIMARY KEY,
                      week_key INTEGER NOT NULL, product_key INTEGER NOT NULL,
                      sales_units NUMERIC NOT NULL, cost_of_goods_sold_cogs NUMERIC, inventory_level NUMERIC,
                      demand_forecast NUMERIC, price_per_unit NUMERIC, gross_profit NUMERIC, operating_profit NUMERIC, revenue NUMERIC,
                      inventory_turnover_ratio NUMERIC, gross_profit_margin NUMERIC, revenue_growth NUMERIC,
                      modified_row BOOLEAN NOT NULL DEFAULT FALSE, run_id VARCHAR(50) NOT NULL,
                      FOREIGN KEY (week_key) REFERENCES warehouse.dim_week(week_key),
                      FOREIGN KEY (product_key) REFERENCES warehouse.dim_product(product_key),
                      FOREIGN KEY (run_id) REFERENCES etl.run_table(run_id),
                      UNIQUE ( week_key, product_key, run_id )
                      )
                      """))

In [47]:
# ============================================================
# 7. WAREHOUSE LAYER
# ============================================================

# warehouse_df = staging_df.copy()
warehouse_df = staging_df.drop( columns=["run_id"] ).copy()

In [48]:

# ------------------------------------------------------------
# Remove rows where Sales_Units is missing
# ------------------------------------------------------------

before_sales_filter = len(warehouse_df)

warehouse_df = warehouse_df[
    warehouse_df["sales_units"].notna()
].copy()

In [49]:
removed_sales_rows = (
    before_sales_filter - len(warehouse_df)
)

print(
    f"Removed {removed_sales_rows} rows "
    f"because Sales_Units was missing."
)

Removed 16 rows because Sales_Units was missing.


In [50]:
# ------------------------------------------------------------
# Track whether each row contains missing values
# BEFORE imputation.
# ------------------------------------------------------------



# ------------------------------------------------------------
# Median imputation
# ------------------------------------------------------------

imputation_columns = [
    "cost_of_goods_sold_cogs",
    "inventory_level",
    "demand_forecast",
    "price_per_unit",
    "gross_profit",
    "operating_profit",
    "revenue",
    "inventory_turnover_ratio",
    "gross_profit_margin",
    "revenue_growth"
]

warehouse_df["modified_row"] = (
    warehouse_df.isna().any(axis=1)
)


In [51]:
median_values = {}
for column in imputation_columns:

    median_value = warehouse_df[column].median()
    median_values[column] = median_value
    warehouse_df[column] = warehouse_df[column].fillna(
        median_value
    )



In [52]:
print("\nMedian values used for imputation:")
for column, value in median_values.items():
    print(f"{column}: {value}")


Median values used for imputation:
cost_of_goods_sold_cogs: 23.009999999999998
inventory_level: 2500.0
demand_forecast: 2066.0
price_per_unit: 29.505000000000003
gross_profit: 6737.075
operating_profit: 3527.765
revenue: 32790.0
inventory_turnover_ratio: 0.65
gross_profit_margin: 0.20545
revenue_growth: 0.0


In [53]:
# 10. LOAD PRODUCT DIMENSION
products = ( warehouse_df[["product"]] .dropna() .drop_duplicates() )

In [54]:
with engine.begin() as conn:
    for product in products["product"]:
        conn.execute(
            text(""" INSERT INTO warehouse.dim_product (product) VALUES (:product) ON CONFLICT (product) DO NOTHING """),
            { "product": str(product) }
        )

In [55]:
# 11. LOAD WEEK DIMENSION
weeks = ( warehouse_df[["week"]] .dropna() .drop_duplicates() )
with engine.begin() as conn:
    for week in weeks["week"]:
        week = int(week)
        # Week 1 = 2020-01-01
        # Each following week = +7 days
        start_date = ( pd.Timestamp("2020-01-01") + pd.Timedelta(days=(week - 1) * 7) ).date()

        conn.execute(
            text(""" INSERT INTO warehouse.dim_week ( week_key, week, start_date )
                 VALUES ( :week_key, :week, :start_date )
                 ON CONFLICT (week_key)
                 DO UPDATE SET week = EXCLUDED.week, start_date = EXCLUDED.start_date
                 """),
                 { "week_key": week, "week": week, "start_date": start_date, }
        )

In [56]:
product_keys = pd.read_sql(
    """ SELECT product_key, product FROM warehouse.dim_product """, engine,
      )

In [58]:
week_keys = pd.read_sql( """ SELECT week_key, week FROM warehouse.dim_week """, engine, )

In [59]:
# 13. BUILD FACT TABLE
fact_df = warehouse_df.merge( product_keys, on="product", how="left", )
fact_df = fact_df.merge( week_keys, on="week", how="left", )

In [60]:
fact_df

,week,product,sales_units,cost_of_goods_sold_cogs,inventory_level,demand_forecast,price_per_unit,gross_profit,operating_profit,revenue,inventory_turnover_ratio,gross_profit_margin,revenue_growth,modified_row,product_key,week_key
0,1,Rice - Basmati,1153.0,21.84,2915,1242.0,29.29,8589.85,5362.48,33771.37,0.28,0.2544,0.0000,False,1,1
1,2,Rice - Basmati,1108.0,21.69,3307,1149.0,27.30,6215.88,3593.65,30248.40,0.38,0.2055,-0.1043,False,1,2
2,3,Rice - Basmati,1527.0,22.38,3280,1639.0,27.48,7787.70,2713.53,41961.96,0.46,0.1856,0.3872,False,1,3
3,4,Rice - Basmati,1299.0,21.41,1981,1093.0,26.56,6689.85,2756.27,34501.44,0.40,0.1939,-0.1778,False,1,4
4,5,Rice - Basmati,1346.0,21.69,3135,1424.0,26.27,6164.68,1804.61,35359.42,0.68,0.1743,0.0249,False,1,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7779,256,Spices - Turmeric,543.0,6.83,197148,574.0,8.46,885.09,357.85,4593.78,0.00,0.1927,-0.0344,False,30,256
7780,257,Spices - Turmeric,573.0,7.07,198075,576.0,8.97,1088.70,546.96,5139.81,0.00,0.2118,0.1189,False,30,257
7781,258,Spices - Turmeric,497.0,7.10,199078,587.0,8.91,899.57,496.50,4428.27,0.00,0.2031,-0.1384,False,30,258
7782,259,Spices - Turmeric,599.0,6.85,199979,505.0,8.55,1018.30,548.63,5121.45,0.00,0.1988,0.1565,False,30,259


In [61]:
# Check that all dimension keys were found
if fact_df["product_key"].isna().any(): 
    raise ValueError( "Some products could not be matched to dim_product." )
if fact_df["week_key"].isna().any():
    raise ValueError( "Some weeks could not be matched to dim_week." )

# Add current ETL run ID
fact_df["run_id"] = run_id

In [62]:
# 14. SELECT FACT COLUMNS
fact_df = fact_df[ [ "week_key", "product_key", "sales_units", "cost_of_goods_sold_cogs", "inventory_level"
                    , "demand_forecast", "price_per_unit", "gross_profit", "operating_profit", "revenue"
                    , "inventory_turnover_ratio", "gross_profit_margin", "revenue_growth", "modified_row", "run_id" ] ]

In [64]:
# dtypes={

# }
fact_df.to_sql(name='sales_data', con=engine, schema='warehouse', if_exists='append'
               , index=False, chunksize=1000
            #    , dtype=dtypes
               )

7784

In [66]:
# 16. CREATE READABLE WAREHOUSE VIEW
with engine.begin() as conn:
    conn.execute(text("""
CREATE OR REPLACE VIEW warehouse.vw_warehouse_sales AS
                      SELECT f.sales_key,
                      -- Dimension attributes
                      w.week, w.start_date, p.product,

                      -- Sales measures
                      f.sales_units, f.cost_of_goods_sold_cogs, f.inventory_level, f.demand_forecast,
                      f.price_per_unit, f.gross_profit, f.operating_profit, f.revenue
                      , f.inventory_turnover_ratio, f.gross_profit_margin, f.revenue_growth,

                      -- ETL information
                      f.modified_row, f.run_id

                      FROM warehouse.sales_data AS f
                      INNER JOIN warehouse.dim_week AS w ON f.week_key = w.week_key
                      INNER JOIN warehouse.dim_product AS p ON f.product_key = p.product_key

                      """))

In [63]:
fact_records = fact_df.to_dict(orient="records")

In [ ]:
with engine.begin() as conn:
    

In [45]:
warehouse_df["run_id"] = run_id

In [ ]:
# ------------------------------------------------------------
# Create warehouse table
# ------------------------------------------------------------

warehouse_df.to_sql(
    name="sales_data",
    con=engine,
    schema="warehouse",
    if_exists="replace",
    index=False,
    dtype={
        "week": Integer(),
        "product": String(100),
        "sales_units": Numeric(),
        "cost_of_goods_sold_cogs": Numeric(),
        "inventory_level": Numeric(),
        "demand_forecast": Numeric(),
        "price_per_unit": Numeric(),
        "gross_profit": Numeric(),
        "operating_profit": Numeric(),
        "revenue": Numeric(),
        "inventory_turnover_ratio": Numeric(),
        "gross_profit_margin": Numeric(),
        "revenue_growth": Numeric(),
        "modified_row": Boolean(),
        "run_id": String(50)
    }
)

print("Warehouse layer loaded successfully.")
print("Warehouse rows:", len(warehouse_df))

In [ ]:
with engine.begin() as conn:

    # --------------------------------------------------------
    # Product dimension
    # --------------------------------------------------------

    conn.execute(text("""
        DROP TABLE IF EXISTS warehouse.fact_sales;
        DROP TABLE IF EXISTS warehouse.dim_product;
        DROP TABLE IF EXISTS warehouse.dim_week;
    """))

    conn.execute(text("""
        CREATE TABLE warehouse.dim_product (
            product_key SERIAL PRIMARY KEY,
            product VARCHAR(100) NOT NULL UNIQUE
        );
    """))

    # --------------------------------------------------------
    # Week dimension
    # --------------------------------------------------------

    conn.execute(text("""
        CREATE TABLE warehouse.dim_week (
            week_key INTEGER PRIMARY KEY,
            week INTEGER NOT NULL UNIQUE
        );
    """))

    # --------------------------------------------------------
    # Fact table
    # --------------------------------------------------------

    conn.execute(text("""
        CREATE TABLE warehouse.fact_sales (

            sales_key SERIAL PRIMARY KEY,

            week_key INTEGER NOT NULL,
            product_key INTEGER NOT NULL,

            sales_units NUMERIC NOT NULL,
            cost_of_goods_sold_cogs NUMERIC,
            inventory_level NUMERIC,
            demand_forecast NUMERIC,
            price_per_unit NUMERIC,
            gross_profit NUMERIC,
            operating_profit NUMERIC,
            revenue NUMERIC,
            inventory_turnover_ratio NUMERIC,
            gross_profit_margin NUMERIC,
            revenue_growth NUMERIC,

            modified_row BOOLEAN NOT NULL DEFAULT FALSE,

            CONSTRAINT fk_fact_week
                FOREIGN KEY (week_key)
                REFERENCES warehouse.dim_week(week_key),

            CONSTRAINT fk_fact_product
                FOREIGN KEY (product_key)
                REFERENCES warehouse.dim_product(product_key),

            CONSTRAINT uq_fact_week_product
                UNIQUE (week_key, product_key)
        );
    """))


# ============================================================
# 9. POPULATE DIM_PRODUCT
# ============================================================

product_df = (
    warehouse_df[["product"]]
    .drop_duplicates()
    .sort_values("product")
    .reset_index(drop=True)
)

product_df.to_sql(
    name="dim_product",
    con=engine,
    schema="warehouse",
    if_exists="append",
    index=False,
    dtype={
        "product": String(100)
    }
)


In [ ]:

# ============================================================
# 10. POPULATE DIM_WEEK
# ============================================================

week_df = (
    warehouse_df[["week"]]
    .drop_duplicates()
    .sort_values("week")
    .reset_index(drop=True)
)

week_df.to_sql(
    name="dim_week",
    con=engine,
    schema="warehouse",
    if_exists="append",
    index=False,
    dtype={
        "week": Integer()
    }
)


# ============================================================
# 11. GET DIMENSION KEYS
# ============================================================

product_keys = pd.read_sql(
    """
    SELECT product_key, product
    FROM warehouse.dim_product
    """,
    engine
)

week_keys = pd.read_sql(
    """
    SELECT week_key, week
    FROM warehouse.dim_week
    """,
    engine
)


In [ ]:

# ============================================================
# 12. BUILD FACT TABLE DATA
# ============================================================

fact_df = warehouse_df.merge(
    product_keys,
    on="product",
    how="left"
)

fact_df = fact_df.merge(
    week_keys,
    on="week",
    how="left"
)


fact_columns = [
    "week_key",
    "product_key",
    "sales_units",
    "cost_of_goods_sold_cogs",
    "inventory_level",
    "demand_forecast",
    "price_per_unit",
    "gross_profit",
    "operating_profit",
    "revenue",
    "inventory_turnover_ratio",
    "gross_profit_margin",
    "revenue_growth",
    "modified_row"
]

fact_df = fact_df[fact_columns]


# ============================================================
# 13. LOAD FACT TABLE
# ============================================================


In [ ]:
fact_df.to_sql(
    name="fact_sales",
    con=engine,
    schema="warehouse",
    if_exists="append",
    index=False,
    dtype={
        "week_key": Integer(),
        "product_key": Integer(),
        "sales_units": Numeric(),
        "cost_of_goods_sold_cogs": Numeric(),
        "inventory_level": Numeric(),
        "demand_forecast": Numeric(),
        "price_per_unit": Numeric(),
        "gross_profit": Numeric(),
        "operating_profit": Numeric(),
        "revenue": Numeric(),
        "inventory_turnover_ratio": Numeric(),
        "gross_profit_margin": Numeric(),
        "revenue_growth": Numeric(),
        "modified_row": Boolean()
    }
)


print("Fact table loaded successfully.")


In [ ]:

# ============================================================
# 14. FINAL VALIDATION
# ============================================================

with engine.connect() as conn:

    raw_count = conn.execute(text("""
        SELECT COUNT(*)
        FROM raw.sales_data
    """)).scalar()

    staging_count = conn.execute(text("""
        SELECT COUNT(*)
        FROM staging.sales_data
    """)).scalar()

    warehouse_count = conn.execute(text("""
        SELECT COUNT(*)
        FROM warehouse.sales_data
    """)).scalar()

    fact_count = conn.execute(text("""
        SELECT COUNT(*)
        FROM warehouse.fact_sales
    """)).scalar()

    modified_count = conn.execute(text("""
        SELECT COUNT(*)
        FROM warehouse.sales_data
        WHERE modified_row = TRUE
    """)).scalar()


print("\n==============================")
print("ETL COMPLETE")
print("==============================")
print(f"Raw rows:       {raw_count}")
print(f"Staging rows:   {staging_count}")
print(f"Warehouse rows: {warehouse_count}")
print(f"Fact rows:      {fact_count}")
print(f"Modified rows:  {modified_count}")